In [1]:
!git clone -b KhangHy https://github.com/PhamQuocNam/CS338.git /kaggle/working/CS338
%cd /kaggle/working/CS338/SpikeGPT
!pip install -r requirements.txt
!pip install tomli fissix
!pip install deepspeed
!sed -i 's/from lib2to3.pgen2 import token/import token/' src/binidx.py

!pip install fastapi uvicorn pyngrok nest-asyncio pydantic wikipedia supabase

Cloning into '/kaggle/working/CS338'...
remote: Enumerating objects: 416, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 416 (delta 66), reused 116 (delta 47), pack-reused 275 (from 1)
Receiving objects: 100% (416/416), 49.10 MiB | 18.46 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/kaggle/working/CS338/SpikeGPT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 kB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.3 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 4.0 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.19.0-py3-none-any.whl size=1843046 sha256=608540d2276c43039759d7e04a1cdd9b32960fda74b67a40cf0202c4e20d207b
  Stored in directory: /root/.cache/pip/wheels/e9/b8/60/ae293108c520a820

# Demo Web App
Demo web app lấy kaggle làm chạy backend

In [2]:
import os
import sys
import gc
import json
import re
import time
import math
import wikipedia
import torch
import torch.nn.functional as F
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import types
import random
from supabase import create_client, Client
from contextlib import asynccontextmanager

# --- CẤU HÌNH SUPABASE ---
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
SUPABASE_KEY = user_secrets.get_secret("SUPABASE_KEY")
SUPABASE_URL = user_secrets.get_secret("SUPABASE_URL")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

In [3]:
print("=== BƯỚC 1: KHỞI TẠO VÀ TẢI 6 MÔ HÌNH ===")

# --- GPT2: Dùng HuggingFace ---
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tok_gpt2 = GPT2Tokenizer.from_pretrained("/kaggle/input/models/hykhangg/gpt2-smallnmedium/transformers/default/1/final_gpt2_small_agent")
if tok_gpt2.pad_token is None:
    tok_gpt2.pad_token = tok_gpt2.eos_token

print("Đang tải GPT2 Small Finetune (124M)...")
model_gpt2_small = GPT2LMHeadModel.from_pretrained("/kaggle/input/models/hykhangg/gpt2-smallnmedium/transformers/default/1/final_gpt2_small_agent").to("cuda").eval()

print("Đang tải GPT2 Medium Finetune (355M)...")
model_gpt2_med = GPT2LMHeadModel.from_pretrained("/kaggle/input/models/hykhangg/gpt2-smallnmedium/transformers/default/1/gpt2-medium-agent").to("cuda").eval()

# --- SpikeGPT: Dùng GPT/GPTConfig + functional.reset_net ---
import importlib
os.chdir('/kaggle/working/CS338/SpikeGPT')
sys.path.insert(0, '/kaggle/working/CS338/SpikeGPT')
os.environ["RWKV_RUN_DEVICE"] = "cuda"
os.environ["RWKV_JIT_ON"] = '1'

import src.model as spikegpt_module
from src.utils import TOKENIZER
from src.spikingjelly.clock_driven import functional

def load_spikegpt(model_path, head_qk_dim):
    """Bản load SpikeGPT đầy đủ nhất - Vá lỗi head_q, head_k và copy_mask."""
    os.environ["RWKV_HEAD_QK_DIM"] = str(head_qk_dim)
    importlib.reload(spikegpt_module)
    GPT = spikegpt_module.GPT
    GPTConfig = spikegpt_module.GPTConfig
    config = GPTConfig(vocab_size=50277, ctx_len=1024, model_type='RWKV', n_layer=18, n_embd=768)
    model = GPT(config)
    
    if head_qk_dim == 0:
        # Giả lập head_q, head_k
        model.head_q = lambda x: torch.zeros((*x.shape[:-1], 0), device=x.device)
        model.head_k = lambda x: torch.zeros((*x.shape[:-1], 0), device=x.device)
        # Giả lập copy_mask (trả về tensor toàn 1 để không làm thay đổi kết quả phép nhân)
        model.copy_mask = torch.ones((1024, 1024), device='cuda')
    
    w = torch.load(model_path, map_location='cpu')
    model.load_state_dict(w, strict=False)
    return model.cuda().eval()



print("Đang tải SpikeGPT Scratch Ep78 WITH HeadQK...")
model_spike_ep78_hq   = load_spikegpt("/kaggle/input/models/jakhanh1/spikegpt-agency-final/pytorch/default/1/Scratch78(best).pth",  head_qk_dim=256)

print("Đang tải SpikeGPT Scratch Ep78 WITHOUT HeadQK...")
model_spike_ep78_nohq = load_spikegpt("/kaggle/input/models/jakhanh1/spikegpt-agency-final/pytorch/default/1/Nohead78(best).pth",  head_qk_dim=0)

print("Đang tải SpikeGPT Scratch Ep220 WITH HeadQK...")
model_spike_ep220_hq  = load_spikegpt("/kaggle/input/models/jakhanh1/spikegpt-agency-final/pytorch/default/1/Scratch220(best).pth", head_qk_dim=256)

print("Đang tải SpikeGPT Finetune Ep220 WITH HeadQK...")
model_spike_ft220_hq  = load_spikegpt("/kaggle/input/models/jakhanh1/spikegpt-agency-final/pytorch/default/1/Finetune220(best).pth",head_qk_dim=256)

# Tokenizer SpikeGPT (dùng chung cho cả 4 model SpikeGPT)
tok_spike = TOKENIZER(
    ["/kaggle/working/CS338/SpikeGPT/20B_tokenizer.json",
     "/kaggle/working/CS338/SpikeGPT/20B_tokenizer.json"],
    UNKNOWN_CHAR=None
)

print("Đã tải xong TẤT CẢ 6 mô hình!")

=== BƯỚC 1: KHỞI TẠO VÀ TẢI 6 MÔ HÌNH ===
Đang tải GPT2 Small Finetune (124M)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Đang tải GPT2 Medium Finetune (355M)...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

/kaggle/working/CS338/SpikeGPT/src/spikingjelly/clock_driven/surrogate.py:888: SyntaxWarning: invalid escape sequence '\p'
  g'(x) = \\frac{\\alpha}{\\sqrt{\pi}}e^{-\\alpha^2x^2}
/kaggle/working/CS338/SpikeGPT/src/spikingjelly/clock_driven/surrogate.py:1291: SyntaxWarning: invalid escape sequence '\g'
  \\beta (x + 1), x \ge 0
2026-05-29 06:58:45.904605: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780037926.087958      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780037926.140304      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780037926.574577      57 computation_placer.cc:177] computation placer already registered. Plea


RWKV_HEAD_QK_DIM 256

[1/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output wkv_cuda.cuda.o.d -DTORCH_EXTENSION_NAME=wkv -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options '-fPIC' -res-usage --maxrregcount 60 --use_fast_math -O3 -Xptxas -O3 -DTmax=1024 -std=c++17 -c /kaggle/working/CS338/SpikeGPT/cuda/wkv_cuda.cu -o wkv_cuda.cuda.o 
ptxas info    : Overriding maximum register limit 256 for '_Z15kernel_backwardIfEviiiPKT_S2_S2_S2_S2_PS0_S3_S3_S3_' with  60 of maxrregcount option
ptxas info    : Overriding maximum regis

In [4]:
print("=== BƯỚC 2: KHỞI TẠO CÔNG CỤ VÀ API SERVER ===")

@asynccontextmanager
async def lifespan(app: FastAPI):
    print("API Server đang khởi động và sẵn sàng nhận request!", flush=True)
    yield 
    print("Hệ thống Backend đang tắt.", flush=True)

from fastapi.middleware.cors import CORSMiddleware
# Khởi tạo App (Chỉ cần 1 lần duy nhất)
app = FastAPI(lifespan=lifespan)
# Cấu hình CORS cực kỳ thông thoáng để tránh lỗi Browser
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], 
    allow_credentials=False, # Để False khi dùng allow_origins=["*"]
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"]
)
@app.get("/")
def read_root():
    return {"status": "success", "message": "API Server online!"}
@app.get("/health")
def health_check():
    return {"status": "online"}

=== BƯỚC 2: KHỞI TẠO CÔNG CỤ VÀ API SERVER ===


In [5]:
# --- HÀM THỰC THI TOOL ---
def execute_tool_logic(tool_name, tool_args):
    t_name = tool_name.lower().strip()
    try:
        # 1. EVALUATE / CALCULATOR
        if "evaluate" in t_name or "calculator" in t_name:
            expr = str(tool_args.get("expression", ""))
            allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
            safe_env = {"__builtins__": None, **allowed}
            result = eval(expr.replace("^", "**"), safe_env, {})
            return {"result": round(float(result), 4)}
            
        # 2. WIKIPEDIA
        elif "wiki" in t_name or "retriever" in t_name:
            query = tool_args.get("query", "")
            page = wikipedia.page(query, auto_suggest=True)
            return {"summary": page.summary[:400] + "..."}
            
        # 3. KIỂM TRA TỒN KHO (check_inventory)
        elif "check_inventory" in t_name:
            product_id = tool_args.get("product_id", tool_args.get("item", "")).upper()
            
            # Truy vấn Supabase
            res = supabase.table("products").select("*").eq("product_id", product_id).execute()
            
            if not res.data:
                return {"status": "error", "message": f"Mã sản phẩm {product_id} không tồn tại trong hệ thống."}
                
            item = res.data[0]
            in_stock = item["stock"] > 0
            return {
                "status": "success",
                "product_name": item["name"],
                "stock": item["stock"],
                "price": f"{item['price']:,} VND",
                "message": f"Sản phẩm {item['name']} còn {item['stock']} chiếc." if in_stock else f"Sản phẩm {item['name']} đã hết hàng."
            }
            
        # 4. TẠO ĐƠN HÀNG (create_order)
        elif "create_order" in t_name:
            product_id = tool_args.get("product_id", "").upper()
            
            # Ép kiểu số lượng
            try:
                qty = int(tool_args.get("quantity", 1))
            except ValueError:
                qty = 1
                
            address = tool_args.get("address", "Nhận tại cửa hàng")
            
            # BƯỚC A: Kiểm tra sản phẩm có tồn tại không?
            prod_res = supabase.table("products").select("name, stock, price").eq("product_id", product_id).execute()
            
            if not prod_res.data:
                return {
                    "status": "error", 
                    "message": f"Tạo đơn thất bại. Mã sản phẩm {product_id} không tồn tại trong hệ thống."
                }
                
            prod_data = prod_res.data[0]
            current_stock = prod_data["stock"]
            price = prod_data["price"]
            
            # BƯỚC B: Kiểm tra tồn kho có đủ bán không?
            if current_stock < qty:
                return {
                    "status": "error",
                    "message": f"Tạo đơn thất bại. Sản phẩm {prod_data['name']} chỉ còn {current_stock} chiếc, không đủ để giao {qty} chiếc."
                }
                
            # BƯỚC C: Tiến hành trừ kho và tạo đơn
            order_id = f"ORD-{random.randint(10000, 99999)}"
            total_price = price * qty
            new_stock = current_stock - qty
            
            # Trừ tồn kho trong DB Products
            supabase.table("products").update({"stock": new_stock}).eq("product_id", product_id).execute()
            
            # Lưu đơn hàng vào DB Orders
            supabase.table("orders").insert({
                "order_id": order_id,
                "product_id": product_id,
                "quantity": qty,
                "total_price": total_price,
                "address": address,
                "status": "Chờ xác nhận"
            }).execute()
            
            return {
                "status": "success",
                "order_id": order_id,
                "total_price": f"{total_price:,} VND",
                "message": f"Đã lên đơn thành công {qty}x {prod_data['name']}. Kho đã tự động trừ (Còn lại {new_stock} chiếc)."
            }
            
        # 5. KIỂM TRA TRẠNG THÁI ĐƠN (get_order)
        elif "get_order" in t_name:
            order_id = tool_args.get("order_id", "").upper()
            res = supabase.table("orders").select("*").eq("order_id", order_id).execute()
            
            if not res.data:
                return {"status": "error", "message": f"Không tìm thấy đơn hàng mã {order_id}."}
                
            order = res.data[0]
            return {
                "status": "success",
                "order_id": order["order_id"],
                "current_status": order["status"],
                "address": order["address"],
                "message": f"Đơn hàng {order_id} đang ở trạng thái: {order['status']}."
            }
            
        # 6. HỦY ĐƠN HÀNG (delete_order)
        elif "delete_order" in t_name:
            order_id = tool_args.get("order_id", "").upper()
            
            # Xóa khỏi Supabase
            res = supabase.table("orders").delete().eq("order_id", order_id).execute()
            
            if not res.data:
                return {"status": "error", "message": f"Hủy thất bại. Không tìm thấy đơn hàng {order_id}."}
                
            return {
                "status": "success",
                "message": f"Đã xóa thành công đơn hàng {order_id} khỏi hệ thống."
            }

        # 7. CẬP NHẬT ĐƠN HÀNG (update_order)
        elif "update_order" in t_name:
            order_id = tool_args.get("order_id", "").upper()
            
            # Cập nhật số lượng nếu có yêu cầu
            new_qty = tool_args.get("quantity")
            if new_qty:
                res = supabase.table("orders").update({"quantity": new_qty}).eq("order_id", order_id).execute()
                if not res.data:
                    return {"status": "error", "message": f"Cập nhật thất bại. Đơn {order_id} không tồn tại."}
                return {"status": "success", "message": f"Đã cập nhật số lượng thành {new_qty} cho đơn {order_id}."}
            else:
                return {"status": "success", "message": "Yêu cầu cập nhật đã được ghi nhận (Cần bổ sung chi tiết)."}

        # 8. THỐNG KÊ DOANH THU (revenue_analysis) 
        elif "revenue_analysis" in t_name:
            # Truy vấn lấy tất cả các đơn hàng trừ đơn đã bị hủy
            res = supabase.table("orders").select("total_price, status").neq("status", "Đã hủy").execute()
            
            if not res.data:
                return {"status": "success", "total_orders": 0, "total_revenue": "0 VND", "message": "Chưa có đơn hàng nào."}
            
            # Tính tổng tiền và đếm số đơn
            real_total_orders = len(res.data)
            real_revenue = sum(item['total_price'] for item in res.data if item['total_price'] is not None)
            
            return {
                "status": "success",
                "action": "Trích xuất báo cáo doanh thu",
                "total_valid_orders": real_total_orders,
                "total_revenue": f"{real_revenue:,} VND",
                "message": f"Đã thống kê {real_total_orders} đơn hàng hợp lệ. Tổng doanh thu thực tế là {real_revenue:,} VND."
            }

        # Tool không khớp (Fallback)
        else:
            return {"error": f"Tool '{t_name}' chưa được hệ thống backend hỗ trợ."}
            
    except Exception as e:
        return {"error": f"Lỗi thực thi Database: {str(e)}"}

In [6]:
def parse_and_execute(raw_output, exec_time):
    raw_output = raw_output.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
    payload = {"text": raw_output, "is_tool": False, "tool_name": "", "tool_args": {}, "execution_result": None, "time": f"{exec_time:.2f}s"}
    
    if "<tool_call>" in raw_output:
        payload["is_tool"] = True
        try:
            # Dùng Regex để bóc tách đoạn JSON nằm giữa 2 thẻ <tool_call>
            json_str = re.search(r"<tool_call>(.*?)</tool_call>", raw_output, re.DOTALL).group(1)
            tool_data = json.loads(json_str)
            payload["tool_name"] = tool_data.get("name", "Unknown")
            payload["tool_args"] = tool_data.get("arguments", {})
            
            # GỌI HÀM DATABASE
            payload["execution_result"] = execute_tool_logic(payload["tool_name"], payload["tool_args"])
            
        except Exception as e:
            payload["tool_name"] = "LỖI CÚ PHÁP"
            payload["tool_args"] = {"Lỗi": "Mô hình sinh sai định dạng JSON", "Raw": raw_output}
            payload["execution_result"] = {"error": "Lỗi Parse JSON: " + str(e)}
            
    return payload

In [7]:
# --- HÀM SUY LUẬN ---

def run_gpt2(model, prompt):
    start_t = time.time()
    inputs = tok_gpt2(prompt, return_tensors="pt").to("cuda")
    len_in = inputs.input_ids.shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=150, do_sample=False,
            pad_token_id=tok_gpt2.pad_token_id,
            eos_token_id=tok_gpt2.eos_token_id
        )
    raw = tok_gpt2.decode(out[0][len_in:], skip_special_tokens=False)
    return parse_and_execute(raw, time.time() - start_t)

def run_spikegpt(model, prompt, max_new_tokens=150):
    """Inference O(N²) với functional.reset_net — dùng cho tất cả model SpikeGPT."""
    start_t = time.time()
    
    # Tokenize input
    ctx = tok_spike.tokenizer.encode(prompt)
    out_tokens = []

    for _ in range(max_new_tokens):
        # Chỉ lấy 1024 token gần nhất (Context Window)
        ctx_crop = ctx[-1024:]
        
        # BẮT BUỘC với SNN: Reset trạng thái neuron trước mỗi lần Forward (vì đây là SNN-Inference-O(N2))
        functional.reset_net(model)
        
        # Forward pass
        with torch.no_grad():
            out = model.forward(torch.tensor([ctx_crop], dtype=torch.long).cuda())
        
        # Lấy token có xác suất cao nhất
        logits = out[0, -1]
        token  = int(torch.argmax(logits).item())
        
        # Nếu gặp token kết thúc (0 hoặc các dấu hiệu khác) thì dừng
        if token == 0:
            break
            
        out_tokens.append(token)
        ctx.append(token)
        
        # Giải mã thử để kiểm tra dấu hiệu dừng (tool call hoặc end of message)
        decoded = tok_spike.tokenizer.decode(out_tokens)
        if "</tool_call>" in decoded or "<|im_end|>" in decoded:
            break

    # Giải mã kết quả cuối cùng
    raw_text = tok_spike.tokenizer.decode(out_tokens)
    
    # Parse kết quả để thực thi công cụ (Database/Wiki...) nếu có
    return parse_and_execute(raw_text, time.time() - start_t)


In [8]:
SYSTEM_PROMPT = "Hãy thực hiện theo yêu cầu"

def build_prompt(message):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{message}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# --- MODEL REGISTRY ---
MODEL_REGISTRY = {
    "gpt2_small":    lambda p: run_gpt2(model_gpt2_small,    p),
    "gpt2_medium":   lambda p: run_gpt2(model_gpt2_med,      p),
    "spike_ep78_hq": lambda p: run_spikegpt(model_spike_ep78_hq,   p),
    "spike_ep78_nohq":lambda p: run_spikegpt(model_spike_ep78_nohq, p),
    "spike_ep220_hq":lambda p: run_spikegpt(model_spike_ep220_hq,  p),
    "spike_ft220_hq":lambda p: run_spikegpt(model_spike_ft220_hq,  p),
}

class ChatRequest(BaseModel):
    message: str

class SingleModelRequest(BaseModel):
    message: str
    model_key: str  # một trong MODEL_REGISTRY

@app.post("/api/generate_all")
def api_generate_all(req: ChatRequest):
    """Chạy tất cả 6 model, trả về dict kết quả để frontend so sánh."""
    prompt = build_prompt(req.message)
    return {key: fn(prompt) for key, fn in MODEL_REGISTRY.items()}

@app.post("/api/generate")
def api_generate(req: SingleModelRequest):
    """Chạy một model cụ thể theo model_key."""
    if req.model_key not in MODEL_REGISTRY:
        return {"error": f"model_key '{req.model_key}' không hợp lệ. Chọn: {list(MODEL_REGISTRY.keys())}"}
    prompt = build_prompt(req.message)
    return {req.model_key: MODEL_REGISTRY[req.model_key](prompt)}

@app.post("/api/compare_gpt2")
def api_compare_gpt2(req: ChatRequest):
    """So sánh 2 model GPT2."""
    prompt = build_prompt(req.message)
    return {
        "gpt2_small":  run_gpt2(model_gpt2_small, prompt),
        "gpt2_medium": run_gpt2(model_gpt2_med,   prompt),
    }

@app.post("/api/compare_spike")
def api_compare_spike(req: ChatRequest):
    """So sánh 4 model SpikeGPT."""
    prompt = build_prompt(req.message)
    return {
        "spike_ep78_hq":  run_spikegpt(model_spike_ep78_hq,   prompt),
        "spike_ep78_nohq":run_spikegpt(model_spike_ep78_nohq,  prompt),
        "spike_ep220_hq": run_spikegpt(model_spike_ep220_hq,   prompt),
        "spike_ft220_hq": run_spikegpt(model_spike_ft220_hq,   prompt),
    }

# --- KHỞI CHẠY NGROK & SERVER ---
ngrokAPI = user_secrets.get_secret("ngrok")
ngrok.set_auth_token(ngrokAPI)
public_url = ngrok.connect(8000).public_url
print(f"THÀNH CÔNG! API URL: {public_url}")
print(f"Endpoints: /api/generate_all | /api/generate | /api/compare_gpt2 | /api/compare_spike")

THÀNH CÔNG! API URL: https://unmoldered-patellate-angela.ngrok-free.dev                             
Endpoints: /api/generate_all | /api/generate | /api/compare_gpt2 | /api/compare_spike


In [9]:
import threading

def run_server():
    # Chạy uvicorn bên trong một luồng hoàn toàn độc lập
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Khởi tạo và chạy luồng ngầm
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Hệ thống Server Backend đã khởi động và đang chạy!")

Hệ thống Server Backend đã khởi động và đang chạy!


INFO:     Started server process [57]
INFO:     Waiting for application startup.


API Server đang khởi động và sẵn sàng nhận request!


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "OPTIONS /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "POST /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "POST /api/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:1b34:b86a:2557:fec7:724:6b48:0 - "POST /api/generate HTTP/1.1" 200 OK
INFO:     200